# GLiNER2 한국어 미세조정

본 노트북은 공개 한국어 NER 데이터셋 [KLUE-NER](https://klue-benchmark.com/)로 GLiNER2.5 multi 체크포인트에 LoRA 어댑터를 학습하고, 학습 전후의 한국어 엔티티 추출 성능을 비교한다.

| 단계 | 내용 |
|---|---|
| 데이터 | KLUE-NER의 문자 단위 BIO 태그를 GLiNER2 학습 형식(원문 위치가 있는 엔티티)으로 변환한다. |
| 분할기 정합성 | 정답 범위가 단어 분할기의 토큰 경계와 맞는지 확인한다. 맞지 않는 범위는 학습할 수 없다. |
| 기준선 | 학습 전 모델을 공백 분할기와 형태소 분할기로 평가한다. |
| 학습 | 형태소 분할기를 설정한 상태로 LoRA 어댑터를 학습한다. |
| 평가 | 어댑터를 불러와 같은 평가 데이터로 다시 평가하고, 직접 만든 문장으로 결과를 확인한다. |

전체 실행에는 CPU 기준 약 10분이 걸린다(체크포인트·데이터 다운로드 제외). 학습 크기는 준비 단계의 상수로 조절한다. 학습한 어댑터는 `outputs/`에 저장되며 저장소에는 포함하지 않는다.

## 결과 요약

> 아래 수치는 2026-11-11 실행 결과이다(학습 2,000문장 × 2 epoch, 평가 500문장·정답 엔티티 806개, Darwin arm64 CPU).

| 조건 | person F1 | organization F1 | location F1 | 전체 P / R / F1 | 조사 포함 예측 |
|---|---|---|---|---|---|
| 학습 전 / 공백 분할 | 0.316 | 0.271 | 0.289 | 0.290 / 0.308 / **0.299** | 453 |
| 학습 전 / 형태소 분할 | 0.572 | 0.602 | 0.603 | 0.541 / 0.640 / **0.586** | 7 |
| **LoRA / 형태소 분할** | **0.856** | **0.780** | **0.733** | 0.821 / 0.806 / **0.814** | 8 |

- **분할기 교체만으로 F1이 두 배가 된다(0.299 → 0.586).** 공백 분할로는 KLUE 정답 범위의 43.4%만 표현할 수 있고, 형태소 분할은 97.2%를 표현한다.
- **LoRA 학습으로 F1이 0.586 → 0.814로 오른다.** 학습 파라미터는 전체의 0.94%(약 270만 개), 어댑터 크기는 10.4MB, 학습 시간은 CPU에서 약 3분 40초이다.
- 향상에는 KLUE의 표기 규칙을 익힌 효과가 포함된다. KLUE 학습 데이터에서 사람 이름 뒤의 직함(`회장`, `대표`, `장관` 등)은 604건 중 584건(97%)이 이름 범위 밖에 표시되어 있고, 학습 후 모델도 `이재용 회장` 대신 `이재용`을 고른다. 직접 만든 문장 7개에서 회귀는 없었지만, 그 7개는 학습 전에도 대부분 맞힌 쉬운 문장이다.

## 준비

### 경고 출력 설정

체크포인트를 불러올 때 출력되는 경고를 숨긴다. 경고의 원인을 처리하지는 않는다.

| 경고 | 출처 | 의미 |
|---|---|---|
| `Redirects are currently not supported in MacOs` | PyTorch 분산 실행 모듈 | macOS에서 출력 리디렉션을 지원하지 않는다는 안내. 이 노트북과 무관하다. |
| `You are sending unauthenticated requests to the HF Hub` | Hugging Face Hub | 토큰 없이 내려받으면 속도 제한이 낮다는 안내. 공개 체크포인트라 토큰 없이도 받을 수 있다. |
| `You are using a model of type gliner2 to instantiate a model of type extractor` | Transformers | GLiNER2 체크포인트를 `gliner2` 패키지의 클래스로 불러올 때 나오는 형식 불일치 안내. 패키지가 의도한 동작이다. |

Transformers와 Hugging Face Hub는 처음 import될 때 자체 로그 수준을 설정하므로, `logging.getLogger(...).setLevel()`이 아니라 각 라이브러리의 `set_verbosity_error()`로 수준을 낮춘다.

In [ ]:
import logging
import warnings

warnings.filterwarnings("ignore", category=UserWarning)
logging.getLogger("torch.distributed.elastic").setLevel(logging.ERROR)

from huggingface_hub.utils import logging as hf_logging
from transformers.utils import logging as transformers_logging

hf_logging.set_verbosity_error()
transformers_logging.set_verbosity_error()

### 형태소 분할기

한국어 형태소 분할기는 프로젝트의 [`config/extractors.py`](../config/extractors.py)에서 가져온다. 프로젝트 루트가 Python 모듈 검색 경로에 포함되어야 한다.

In [ ]:
from config.extractors import kiwi_splitter

### 실험 크기

| 상수 | 의미 |
|---|---|
| `TRAIN_SIZE` | KLUE-NER 학습 데이터(21,008문장)에서 무작위로 뽑을 문장 수 |
| `EVAL_SIZE` | KLUE-NER 검증 데이터(5,000문장)에서 무작위로 뽑을 평가 문장 수 |
| `EPOCHS`, `BATCH_SIZE` | 학습 반복 수와 배치 크기 |

LoRA 학습은 CPU에서 초당 10문장 안팎으로 진행된다. 값을 키우면 성능이 오를 수 있지만 시간도 비례해서 늘어난다.

In [ ]:
from pathlib import Path

TRAIN_SIZE = 2000
EVAL_SIZE = 500
EPOCHS = 2
BATCH_SIZE = 8
SEED = 42
BASE_MODEL = "fastino/gliner2.5-multi-v1"
ADAPTER_DIR = Path("outputs/klue_ner_lora")  # 프로젝트 루트 기준

## 데이터

### KLUE-NER 불러오기

KLUE-NER(CC BY-SA 4.0)는 한국어 뉴스 기사(WIKITREE)와 영화 리뷰(NSMC) 문장에 6종의 엔티티를 **문자 단위** BIO 태그로 표시한 데이터셋이다([개념과 동작 구조](00_gliner2_overview.ipynb)의 학습 절 참고). Hugging Face의 `klue/klue` 저장소에서 parquet 파일을 내려받는다.

| KLUE 태그 | 의미 | 이 노트북의 레이블 |
|---|---|---|
| `PS` | 사람 | `person` |
| `OG` | 기관 | `organization` |
| `LC` | 장소 | `location` |
| `DT`, `TI`, `QT` | 날짜, 시간, 수량 | 사용하지 않음 |

POLE+O 온톨로지와 겹치는 세 타입만 학습한다. 날짜·시간·수량 범위는 정답에서 제외되므로, 학습 중에는 "추출하지 않아야 할 범위"로 취급된다.

In [ ]:
import json
import random

import pyarrow.parquet as pq
from huggingface_hub import hf_hub_download

KLUE_TO_LABEL = {"PS": "person", "OG": "organization", "LC": "location"}
LABELS = list(KLUE_TO_LABEL.values())


def load_klue(split):
    path = hf_hub_download("klue/klue", f"ner/{split}-00000-of-00001.parquet", repo_type="dataset")
    table = pq.read_table(path)
    features = json.loads(table.schema.metadata[b"huggingface"])["info"]["features"]
    return table.to_pylist(), features["ner_tags"]["feature"]["names"]


def to_example(row, tag_names):
    """Character BIO tags -> (text, [(start, end, label)])."""
    text = "".join(row["tokens"])
    tags = [tag_names[t] for t in row["ner_tags"]] + ["O"]
    spans, start, kind = [], None, None
    for index, tag in enumerate(tags):
        if start is not None and not tag.startswith("I-"):
            if kind in KLUE_TO_LABEL:
                spans.append((start, index, KLUE_TO_LABEL[kind]))
            start = None
        if tag.startswith("B-"):
            start, kind = index, tag[2:]
    return text, spans


train_rows, tag_names = load_klue("train")
eval_rows, _ = load_klue("validation")
random.seed(SEED)
train_data = [to_example(row, tag_names) for row in random.sample(train_rows, TRAIN_SIZE)]
eval_data = [to_example(row, tag_names) for row in random.sample(eval_rows, EVAL_SIZE)]

print(f"tags: {tag_names}")
print(f"train={len(train_data)} eval={len(eval_data)} "
      f"eval entities={sum(len(spans) for _, spans in eval_data)}")
text, spans = train_data[0]
print(text)
print([(text[s:e], label) for s, e, label in spans])

첫 학습 문장의 정답은 `CBS`(organization), `김현정`·`김진표`·`남경필`·`김진표`(person)이다. `김진표`가 두 번 나오므로 원문 위치로 구분해야 한다. 평가 500문장에는 세 타입의 정답 엔티티가 806개 있다.

### 분할기 정합성

GLiNER2는 단어 토큰의 경계에서만 범위를 만들 수 있다. 정답 범위의 시작과 끝이 분할기가 만든 토큰 경계와 일치하는 비율을 분할기별로 센다. 일치하지 않는 범위는 그 분할기로는 예측할 수도, 학습할 수도 없다.

In [ ]:
from gliner2.processing.word_splitter import WhitespaceTokenSplitter

SPLITTERS = {"whitespace": WhitespaceTokenSplitter(), "kiwi": kiwi_splitter}


def alignment(data, splitter):
    aligned = total = 0
    examples = []
    for text, spans in data:
        starts, ends = set(), set()
        for _, start, end in splitter(text, lower=False):
            starts.add(start)
            ends.add(end)
        for start, end, label in spans:
            total += 1
            if start in starts and end in ends:
                aligned += 1
            elif len(examples) < 3:
                examples.append(text[max(0, start - 3):end + 4].replace(text[start:end], f"[{text[start:end]}]"))
    return aligned, total, examples


for name, splitter in SPLITTERS.items():
    aligned, total, examples = alignment(train_data, splitter)
    print(f"{name:>10}: {aligned}/{total} ({aligned / total:.1%}) aligned   e.g. {examples}")

| 분할기 | 경계가 맞는 정답 범위 | 맞지 않는 예 |
|---|---|---|
| `whitespace` | 1,209 / 2,783 (**43.4%**) | `[김현정]의`, `[윌켄]의`, `[맥라이언]만으로도` |
| `kiwi` | 2,705 / 2,783 (**97.2%**) | `[Che]...`, `[코난]의`, `어린[이정현]` |

공백 분할로는 KLUE 정답 범위의 절반 이상을 표현할 수 없다. 조사가 붙은 이름은 모두 이 경우에 해당한다. 이 상태로는 아무리 학습해도 해당 범위를 예측할 수 없고, 학습 데이터로도 쓸 수 없다. 형태소 분할에서 남는 2.8%는 형태소 분석기가 이름을 다른 경계로 자른 경우(`어린이`+`정현`)나 문장부호가 붙은 영문 등이며, 학습에서 제외한다.

### 학습 형식으로 변환

GLiNER2의 `InputExample`은 레이블별 엔티티 목록을 받는다. 같은 문자열이 문장에 여러 번 나올 수 있으므로 원문 위치(`start`, `end`)를 함께 준다. 형태소 경계와 맞지 않는 범위는 학습에서 제외한다.

모든 예제에 세 레이블을 빈 목록이라도 선언한다. GLiNER2는 예제에 선언된 레이블만 학습하므로, 선언하지 않으면 엔티티가 없는 문장은 학습할 작업이 없는 예제로 거부되고, 일부 레이블만 있는 문장에서는 나머지 레이블의 "없음"을 배우지 못한다. KLUE-NER는 세 타입을 빠짐없이 표시한 데이터이므로 빈 목록이 실제 정답이다.

In [ ]:
from gliner2.training.data import InputExample, TrainingDataset

starts_ends = {}


def boundaries(text):
    if text not in starts_ends:
        tokens = list(kiwi_splitter(text, lower=False))
        starts_ends[text] = ({s for _, s, _ in tokens}, {e for _, _, e in tokens})
    return starts_ends[text]


def to_input_example(text, spans):
    starts, ends = boundaries(text)
    entities = {label: [] for label in LABELS}  # 빈 목록 = 이 문장에 해당 타입이 없다
    for start, end, label in spans:
        if start in starts and end in ends:
            entities[label].append({"text": text[start:end], "start": start, "end": end})
    return InputExample(text=text, entities=entities)


train_examples = [to_input_example(text, spans) for text, spans in train_data]
dataset = TrainingDataset(train_examples)
dataset.validate(raise_on_error=True)  # 엔티티 텍스트가 원문에 실제로 있는지 검사한다
print(f"✓ {len(train_examples)} examples, "
      f"{sum(len(v) for ex in train_examples for v in ex.entities.values())} entities")

## 평가 방법

평가 문장마다 `extract_entities(..., include_spans=True)`로 예측하고, 원문 위치와 레이블이 모두 같은 경우만 정답으로 센다(정확 일치). 레이블별·전체 정밀도(precision), 재현율(recall), F1을 계산한다. 예측 이름이 조사로 끝나는 경우(`JeekLee는`)도 따로 센다.

KLUE-NER는 자체 표기 규칙을 따른다. 예를 들어 `박씨`에서 성 `박`만 사람으로 표시한다. 기준선 모델은 이 규칙을 모르므로 정확 일치 점수가 낮게 나올 수 있으며, 학습 후의 향상에는 표기 규칙을 익힌 효과가 포함된다.

In [ ]:
from collections import Counter
from time import perf_counter

from config.extractors import kiwi


def has_particle(name):
    tokens = kiwi().tokenize(name)
    return bool(tokens) and tokens[-1].tag.startswith("J")


def evaluate(extractor, data, threshold=0.5):
    tp, fp, fn = Counter(), Counter(), Counter()
    particles = 0
    started = perf_counter()
    for text, spans in data:
        result = extractor.extract_entities(text, LABELS, threshold=threshold, include_spans=True)
        predicted = {(item["start"], item["end"], label)
                     for label, items in result["entities"].items() for item in items}
        gold = set(spans)
        particles += sum(has_particle(text[s:e]) for s, e, _ in predicted)
        for _, _, label in predicted & gold:
            tp[label] += 1
        for _, _, label in predicted - gold:
            fp[label] += 1
        for _, _, label in gold - predicted:
            fn[label] += 1
    scores = {}
    for label in [*LABELS, "overall"]:
        t = sum(tp.values()) if label == "overall" else tp[label]
        p = sum(fp.values()) if label == "overall" else fp[label]
        n = sum(fn.values()) if label == "overall" else fn[label]
        precision = t / (t + p) if t + p else 0.0
        recall = t / (t + n) if t + n else 0.0
        f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
        scores[label] = {"precision": precision, "recall": recall, "f1": f1}
    scores["particles"] = particles
    scores["seconds"] = perf_counter() - started
    return scores


def show(results):
    print(f"{'condition':<24}" + "".join(f"{label:>14}" for label in [*LABELS, "overall"])
          + f"{'particles':>11}{'sec':>7}")
    for name, scores in results.items():
        cells = "".join(f"{scores[label]['f1']:>14.3f}" for label in [*LABELS, "overall"])
        print(f"{name:<24}{cells}{scores['particles']:>11}{scores['seconds']:>7.0f}")
    print("(F1 per label; overall precision/recall below)")
    for name, scores in results.items():
        print(f"  {name}: P={scores['overall']['precision']:.3f} R={scores['overall']['recall']:.3f}")

## 기준선

학습 전 multi 체크포인트를 두 분할기로 평가한다.

In [ ]:
from gliner2 import AutoExtractor

model = AutoExtractor.from_pretrained(BASE_MODEL)
results = {}
for name, splitter in (("base / whitespace", "whitespace"), ("base / kiwi", kiwi_splitter)):
    model.set_word_splitter(splitter)
    results[name] = evaluate(model, eval_data)
show(results)

| 조건 | person | organization | location | 전체 F1 | 정밀도 | 재현율 | 조사 포함 예측 |
|---|---|---|---|---|---|---|---|
| 공백 분할 | 0.316 | 0.271 | 0.289 | 0.299 | 0.290 | 0.308 | 453 |
| 형태소 분할 | 0.572 | 0.602 | 0.603 | 0.586 | 0.541 | 0.640 | 7 |

공백 분할의 예측 중 453개가 조사로 끝나며, 모두 정확 일치에서 오답이 된다. 형태소 분할은 학습 없이도 F1을 두 배로 올린다. 남은 오류는 KLUE 표기 규칙과의 범위 차이(`이재용 회장` 대 `이재용`), 타입 오류, 누락이다. 500문장 평가에 각각 18초가 걸렸다(문장당 약 0.04초).

## LoRA 학습

형태소 분할기를 설정한 모델로 LoRA 어댑터를 학습한다. 분할기는 학습 데이터의 수집에도 적용되므로, 학습과 추론에서 같은 분할기를 쓴다.

| 설정 | 값 | 의미 |
|---|---|---|
| `use_lora` | `True` | 인코더와 작업 헤드에 저차원 어댑터만 학습한다. multi 기준 학습 파라미터는 전체의 약 1%이다. |
| `encoder_lr`, `task_lr` | 기본값 | LoRA에서는 어댑터 파라미터에 `task_lr`(5e-4)이 적용된다. |
| `eval_strategy` | `"no"` | 학습 중 평가는 생략하고, 학습 후 위의 평가 함수로 비교한다. |
| `num_workers`, `pin_memory` | `0`, `False` | 노트북 커널과 CPU 환경에서 데이터 로더를 단순하게 유지한다. |

학습이 끝나면 `ADAPTER_DIR/final`에 어댑터가 저장된다.

In [ ]:
from gliner2.training.trainer import ExtractorTrainer, TrainingConfig

model.set_word_splitter(kiwi_splitter)
config = TrainingConfig(
    output_dir=str(ADAPTER_DIR),
    num_epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    use_lora=True,
    save_adapter_only=True,
    eval_strategy="no",
    logging_steps=50,
    num_workers=0,
    pin_memory=False,
    seed=SEED,
)
summary = ExtractorTrainer(model, config).train(train_data=dataset)
print({key: summary[key] for key in ("total_steps", "total_epochs", "total_time_seconds", "samples_per_second")})

| 항목 | 값 |
|---|---|
| 학습 파라미터 | 2,715,664 / 290,070,823 (0.94%) |
| 최적화 단계 | 500 (2,000문장 × 2 epoch ÷ 배치 8), 워밍업 50 |
| 학습 시간 | 219.6초 (초당 18.2문장) |
| 저장 | `outputs/klue_ner_lora/final`, 10.4MB |

## 학습 후 평가

기반 체크포인트를 새로 불러와 저장된 어댑터를 붙인다. 분할기는 저장되지 않으므로 다시 설정한다. 학습에 사용한 모델 객체를 그대로 쓰지 않고 저장·불러오기를 거쳐, 실제 사용과 같은 경로로 평가한다.

In [ ]:
tuned = AutoExtractor.from_pretrained(BASE_MODEL)
tuned.load_adapter(str(ADAPTER_DIR / "final"))
tuned.set_word_splitter(kiwi_splitter)
results["LoRA / kiwi"] = evaluate(tuned, eval_data)
show(results)

| 조건 | person | organization | location | 전체 F1 | 정밀도 | 재현율 | 조사 포함 예측 |
|---|---|---|---|---|---|---|---|
| 학습 전 / 공백 분할 | 0.316 | 0.271 | 0.289 | 0.299 | 0.290 | 0.308 | 453 |
| 학습 전 / 형태소 분할 | 0.572 | 0.602 | 0.603 | 0.586 | 0.541 | 0.640 | 7 |
| **LoRA / 형태소 분할** | **0.856** | **0.780** | **0.733** | **0.814** | **0.821** | **0.806** | 8 |

- 전체 F1이 0.586 → 0.814로 오르고, 정밀도(0.541 → 0.821)가 크게 개선된다. 학습 전에는 정답보다 넓은 범위나 정답 외 엔티티를 많이 예측했다는 뜻이다.
- person의 향상이 가장 크다(0.572 → 0.856). KLUE는 사람 이름 뒤의 직함을 대부분 범위 밖에 두므로(학습 데이터 기준 97%), 이 규칙을 학습한 효과가 포함된다. 다만 오류 유형별 기여도는 분석하지 않았다.
- 어댑터를 붙여도 추론 시간은 크게 늘지 않는다(500문장 18초 → 24초).

### 직접 만든 문장

[단어 분할기 비교](../experiments/02_korean_word_splitter.ipynb)에서 쓴 문장 중 사람·조직·장소가 있는 문장으로 학습 전후 결과를 비교한다. KLUE와 다른 문체(대화체, IT 용어)에서도 효과가 유지되는지 확인하기 위한 정성 비교이다.

In [ ]:
SENTENCES = [
    "이제 GCP에서 매달 나가는 비용은 Cloud DNS 영역 3개, 월 약 $0.6뿐입니다.",
    "오라클 공식 입장: 고정 IP 기능을 처음 발표할 때 'There is no charge to use this feature'라고 밝혔습니다.",
    "삼성전자는 어제 이재용 회장이 서울 서초구 사옥에서 엔비디아와 협력을 발표했다고 밝혔다.",
    "JeekLee 팀장은 다음 주 부산에서 열리는 AWS 서밋에 카카오 개발자들과 함께 참석한다.",
    "네이버클라우드와 KT가 공동으로 판교에 데이터센터를 짓기로 했습니다.",
    "JeekLee는 2023년부터 무신사에서 백엔드 개발자로 일하고 있고, 지금은 성수동에 산다.",
    "쿠팡은 대만 법인을 자회사로 두고 있으며, 본사는 서울 송파구에 있다.",
]
base = AutoExtractor.from_pretrained(BASE_MODEL)
base.set_word_splitter(kiwi_splitter)
for sentence in SENTENCES:
    print(sentence)
    for name, extractor in (("base", base), ("LoRA", tuned)):
        entities = extractor.extract_entities(sentence, LABELS)["entities"]
        print(f"  {name:>5}: " + ", ".join(f"{text}({label[:3]})" for label, items in entities.items()
                                          for text in items))

| 문장 | 학습 전 (형태소 분할) | LoRA | 변화 |
|---|---|---|---|
| GCP … Cloud DNS | GCP(org), **Cloud DNS(loc)** | GCP(org) | 잘못된 타입의 `Cloud DNS`가 빠짐 |
| 오라클 공식 입장 … | 오라클(org) | 오라클(org) | 동일 |
| 삼성전자 … 이재용 회장 … | **이재용 회장**(per), 삼성전자, 엔비디아, 서울 서초구 | **이재용**(per), 삼성전자, 엔비디아, 서울 서초구 | KLUE 규칙대로 직함 제외 |
| JeekLee 팀장 … AWS 서밋 … | JeekLee, 카카오, AWS, 부산 | 동일 | 동일 |
| 네이버클라우드와 KT … | KT, 네이버클라우드, 판교 | 동일 | 동일 |
| JeekLee … 무신사 … 백엔드 개발자 … | JeekLee, 무신사, 성수동 | 동일 | 동일 |
| 쿠팡 … 대만 법인 … | 쿠팡, 서울 송파구, 대만(loc) | 동일 | 동일 |

- 개선: `Cloud DNS(loc)` 오분류가 사라지고, `이재용 회장`이 KLUE 규칙대로 직함을 뺀 `이재용`으로 정리되었다. `JeekLee 팀장`은 학습 전에도 이름만 골랐다.
- 회귀는 보이지 않았지만, 문장 수가 적고 대부분 학습 전에도 맞힌 문장이라 근거로는 약하다. KLUE는 뉴스 기사와 영화 리뷰로 구성되어 IT 서비스명이나 업무 대화는 드물므로, 그런 문장에서의 효과는 따로 확인해야 한다.
- 학습 데이터의 도메인과 표기 규칙이 결과를 좌우한다. 실제 대화 메모리에 적용하려면 대상 도메인의 문장을 학습·평가 데이터에 포함해야 한다.

## 결론과 한계

**결론**

1. 한국어 GLiNER2 학습의 전제 조건은 단어 분할이다. 공백 분할로는 KLUE 정답 범위의 43.4%만 표현할 수 있어, 학습 이전에 데이터 정합성이 깨진다. 형태소 분할기로 학습·추론 모두를 맞추어야 한다.
2. 형태소 분할기만으로 F1이 0.299 → 0.586, LoRA 학습을 더하면 0.814가 된다. 2,000문장과 CPU 4분 미만의 학습으로 얻은 결과이며, 어댑터는 10MB 수준이다.
3. 향상은 학습 데이터의 도메인과 표기 규칙을 따라간다. 직함 제외 같은 규칙은 학습되지만, 학습 데이터에 드문 유형(IT 서비스명, 업무 대화)에서의 효과는 이 노트북으로 검증하지 않았다.

**한계와 다음 단계**

- 평가는 KLUE-NER 검증 데이터 500문장의 사람·조직·장소에 한정된다. 학습 크기, epoch, LoRA 설정은 탐색하지 않았다.
- KLUE-NER에는 관계가 없으므로 관계 추출은 학습하지 않았다. 인코더에 붙은 LoRA가 관계 추출에 미치는 영향도 평가하지 않았다. 관계 학습에는 [KLUE-RE](https://klue-benchmark.com/) 같은 한국어 관계 데이터셋을 검토할 수 있다.
- 대화 메모리 용도라면 실제 대화 문장으로 만든 평가 데이터가 필요하다. 이 경우 [단어 분할기 비교](../experiments/02_korean_word_splitter.ipynb)의 참고 정답을 확장하여 학습 전후와 LLM을 함께 비교하는 실험을 `experiments/`에 추가할 수 있다.